# CS431 · Bổ sung Clean cho CS-B1@0.7 (Kaggle T4)

**Notebook làm gì:** chạy B1 và B-Clean trên **2.262 ảnh Clean**, lưu box + score giống hệt dump LR2 trước đây. Sau đó đánh giá **CS-B1 áp cho mọi ảnh (không định tuyến)** so với B1 `.40/.60`, B1 `epsilon_005/.50` và B-Clean `.40/.60`, trên cả Clean và LR2.

- **Không phải phương pháp mới**, không quét ngưỡng. Spec khóa trước: `docs/research/janek/csb1_clean_check_spec_20261123_janek.md`.
- Dữ liệu là development, nên **không xác nhận** được cải tiến.

### Trước khi chạy
1. Kaggle → **New Dataset**, tải `cs431-csb1-clean.zip` lên (Private).
2. Import notebook này. Bật **GPU T4 x2** (hoặc T4), **Internet On**, **Add Input** chọn dataset vừa tạo.
3. Chạy **Ô 1 → Ô 4**, xem thời gian dự kiến ở Ô 4, rồi chạy **Ô 5 → Ô 7**.

**Thời gian (chưa đo):** cài môi trường 5–40 phút; dump khoảng 55 phút; đánh giá khoảng 10 phút.

## Ô 1 · Cấu hình, tìm dataset, kiểm GPU

Kiểm SHA của mọi file trong dataset. Ghim **GPU 0** (`CUDA_VISIBLE_DEVICES=0`), vì code của Tuấn yêu cầu đúng một GPU.

In [ ]:
# ===================== Ô 1 · CẤU HÌNH =====================
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

PIN = dict(
    MONKEY_V2_URL='https://github.com/ncat218/MONKEY_V2.git',
    MONKEY_V2_COMMIT='dc19b85c8f73d56dd0510165069597363fe25380',
    MONKEYOCRV2_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git',
    MONKEYOCRV2_COMMIT='da96afede92f198c19aca74e1e5f5aa7f4c26af2',
    TORCH='2.9.0', TORCHVISION='0.24.0', TORCHAUDIO='2.9.0', TORCH_INDEX='https://download.pytorch.org/whl/cu128',
    MMENGINE='0.10.7', MMCV='2.0.1', MMOCR='1.0.1', PILLOW='11.3.0', OPENCV='4.10.0.84',
    DB_PP_SHA256='9bfff8aaaca103dd08d7565f2b7a5410ab20a6427fe1f0baad5a5d04d207c8b8',
    BUNDLE_MANIFEST_SHA256='e06ef54e720b43bcb658fd734291fb1b6e4a08db13a8bd4048f16cfd3a2dc573',
)
W = Path('/kaggle/working'); OUT = W/'cs431_out'; LOGS = OUT/'logs'
T = Path('/tmp/cs431')
for p in (OUT, LOGS, T):
    p.mkdir(parents=True, exist_ok=True)

def fail(msg):
    raise SystemExit('LỖI: ' + msg)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

found = [p for p in Path('/kaggle/input').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'csb1_clean']
if not found:
    zips = [z for z in Path('/kaggle/input').glob('**/*.zip') if 'csb1-clean' in z.name.lower()]
    if not zips:
        fail('Không thấy dataset cs431-csb1-clean. Bấm "Add Input" và chọn dataset đó.')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(T/'dataset')
    found = [p for p in (T/'dataset').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'csb1_clean']
if len(found) != 1:
    fail(f'Thấy {len(found)} dataset zoom; chỉ gắn một.')
DS = found[0].parent
manifest = json.loads(found[0].read_text())
bad = [rel for rel, info in manifest['files'].items()
       if rel != 'KAGGLE_MANIFEST.json' and (not (DS/rel).is_file() or sha256(DS/rel) != info['sha256'])]
if bad:
    fail(f'{len(bad)} file thiếu hoặc sai SHA, ví dụ {bad[:3]}.')
print(f'Dataset: {DS} ({len(manifest["files"])} file, SHA khớp) · {manifest["sources"]} nguồn')

CODE = T/'code'
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(DS/'code', CODE)
code_sha = hashlib.sha256((DS/'code/CODE_MANIFEST.json').read_bytes()).hexdigest()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import mixab.colab_helpers_janek as ch
importlib.reload(ch)

gpus = (ch.output_of(['nvidia-smi', '--query-gpu=index,name,driver_version,memory.total', '--format=csv,noheader']) or '').splitlines()
if not gpus:
    fail('Không thấy GPU. Settings -> Accelerator -> GPU T4 x2 (hoặc GPU T4).')
print('GPU của máy:'); [print('  ', g) for g in gpus]
GPU = ch.gpu_label(gpus[0].split(',')[1].strip())
CONFIG_NOTE = f'{len(gpus)} GPU trên máy; chỉ dùng GPU 0 qua CUDA_VISIBLE_DEVICES=0'
print(CONFIG_NOTE)

PY = sys.executable
MV2, MOCR = T/'MONKEY_V2', T/'MonkeyOCRv2'
BUNDLE = MV2/'docs/handoff/artifacts/pr_quad_audit_v1'
WEIGHTS, CLEAN, DUMP, A2R = DS/'weights', DS/'clean', DS/'dump', DS/'a2r'
ENCODER = T/'encoder'
if not (ENCODER/'model.safetensors').is_file():
    shutil.copytree(DS/'encoder', ENCODER, dirs_exist_ok=True)
PREPARED, CONFIG_PATH = T/'prepared', T/'inference_config_kaggle.py'
ENV = dict(MONKEY_V2_ROOT=str(MV2), PYTHONPATH=str(CODE), CUDA_VISIBLE_DEVICES='0')
print(f'Python {sys.version.split()[0]} · code manifest {code_sha[:12]}… · kết quả: {OUT}')

## Ô 2 · Cài môi trường (cùng phiên bản với lần chạy dump T4 trên Colab)

Thứ tự: torch 2.9.0 cu128 → mmengine 0.10.7 → **MMCV 2.0.1** (dùng wheel trong dataset nếu cùng phiên bản Python, nếu không thì **build từ source**) → `install.sh` của MonkeyOCRv2 (MMOCR 1.0.1 + add-on) → ghim Pillow 11.3.0 và OpenCV 4.10.0.84.
- Build MMCV có thể mất **20–40 phút** (ước lượng, chưa đo trên Kaggle). Wheel build xong được chép vào `/kaggle/working/cs431_out/wheels/`: **tải về và thêm vào dataset** để lần sau khỏi build.
- Cần `nvcc` CUDA **12.x** để build. Ô này dùng toolkit có sẵn trên Kaggle (ưu tiên 12.8); nếu không có thì cài `cuda-toolkit-12-8` (không cài driver).
- Kiểm cuối chạy trong tiến trình mới: phiên bản, `MonkeyOCRv2ViTAEBackbone` đã đăng ký, SHA của DBPostprocessor phải đúng bản Tuấn ghim.
- Nếu Kaggle báo cần *Restart*, restart rồi chạy lại **Ô 1** và **Ô 2** (phần đã cài sẽ bỏ qua).

In [ ]:
# ===================== Ô 2 · CÀI MÔI TRƯỜNG =====================
import platform, re
LOG = LOGS/f'install_{GPU}.log'
pyver = sys.version_info
if not ((3, 10) <= pyver[:2] <= (3, 13)):
    print(f'CẢNH BÁO: Python {platform.python_version()} chưa được kiểm với MMCV 2.0.1.')

def ver(name):
    return ch.package_version(name)

def pip(*args):
    ch.run([PY, '-m', 'pip', 'install', '-q', *args], LOG, title='pip install ' + ' '.join(args))

if ver('torch') != f"{PIN['TORCH']}+cu128" or ver('torchvision') != f"{PIN['TORCHVISION']}+cu128":
    pip(f"torch=={PIN['TORCH']}", f"torchvision=={PIN['TORCHVISION']}", '--index-url', PIN['TORCH_INDEX'])
if ver('torchaudio') != f"{PIN['TORCHAUDIO']}+cu128":
    pip('--no-deps', f"torchaudio=={PIN['TORCHAUDIO']}+cu128", '--index-url', PIN['TORCH_INDEX'])
cuda_ok = ch.output_of([PY, '-c', 'import torch;print(torch.version.cuda, torch.cuda.is_available())'], env=ENV)
print('torch CUDA:', cuda_ok)
if not cuda_ok or 'True' not in cuda_ok:
    fail('torch không dùng được CUDA (driver Kaggle quá cũ cho cu128?). Gửi log: ' + str(LOG))
if ver('mmengine') != PIN['MMENGINE']:
    pip(f"mmengine=={PIN['MMENGINE']}")
pip('setuptools<81', 'wheel', 'ninja')

def mmcv_ok():
    if ver('mmcv') != PIN['MMCV']:
        return False
    probe = subprocess.run([PY, '-c', 'from mmcv.ops import nms; print("IMPORT_NMS_OK")'], capture_output=True, text=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nMMCV import probe:\n' + probe.stdout + probe.stderr)
    return probe.returncode == 0 and 'IMPORT_NMS_OK' in probe.stdout.splitlines()

def find_toolkit():
    """CUDA 12.x toolkit with nvcc + cuda.h; prefer 12.8. torch only refuses a different MAJOR version."""
    import glob
    homes = ['/usr/local/cuda-12.8', os.environ.get('CUDA_HOME'), '/usr/local/cuda', *sorted(glob.glob('/usr/local/cuda-12*'))]
    nvcc = shutil.which('nvcc')
    if nvcc:
        homes.append(str(Path(nvcc).resolve().parents[1]))
    best = None
    for home in homes:
        if not home or not (Path(home)/'bin/nvcc').is_file() or not (Path(home)/'include/cuda.h').is_file():
            continue
        m = re.search(r'release (\d+)\.(\d+)', ch.output_of([str(Path(home)/'bin/nvcc'), '--version']) or '')
        if m and m.group(1) == '12':
            cand = (int(m.group(2)) == 8, int(m.group(2)), str(Path(home).resolve()))
            best = max(best, cand) if best else cand
    return best

tag = f'cp{pyver[0]}{pyver[1]}'
if not mmcv_ok():
    wheels = sorted((DS/'wheels').glob(f"mmcv-{PIN['MMCV']}-{tag}-*.whl")) + sorted(OUT.glob(f"wheels/mmcv-{PIN['MMCV']}-{tag}-*.whl"))
    if wheels:
        print('Cài MMCV từ wheel có sẵn:', wheels[0].name)
        pip('--no-deps', str(wheels[0]))
    else:
        tk = find_toolkit()
        if not tk:
            print('Không có CUDA toolkit 12.x. Cài cuda-toolkit-12-8 từ NVIDIA (không cài driver)...')
            distro = platform.freedesktop_os_release()
            repo = 'ubuntu' + distro.get('VERSION_ID', '22.04').replace('.', '')
            from urllib.request import urlretrieve
            keyring = T/'cuda-keyring_1.1-1_all.deb'
            urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/cuda-keyring_1.1-1_all.deb', keyring)
            ch.run(['dpkg', '-i', keyring], LOG)
            ch.run(['apt-get', 'update'], LOG)
            ch.run(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-12-8'], LOG, env={'DEBIAN_FRONTEND': 'noninteractive'})
            tk = find_toolkit()
            if not tk:
                fail('Đã cài nhưng vẫn không thấy nvcc 12.x; xem ' + str(LOG))
        cuda_home = tk[2]
        print(f'Dùng CUDA toolkit {cuda_home} (12.{tk[1]})' + ('' if tk[0] else ' — khác 12.8, torch chỉ cảnh báo khác minor version'))
        src = T/'mmcv_src'
        if not (src/'setup.py').is_file():
            ch.run(['git', 'clone', '-b', f"v{PIN['MMCV']}", '--depth', '1', 'https://github.com/open-mmlab/mmcv.git', src], LOG)
        setup = (src/'setup.py').read_text()
        setup = setup.replace('-std=c++14', '-std=c++17')
        old_v = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
        new_v = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)\n    return namespace['__version__']"
        if old_v in setup:
            setup = setup.replace(old_v, new_v)
        elif new_v not in setup:
            fail('MMCV setup.py khác cấu trúc đã kiểm.')
        (src/'setup.py').write_text(setup)
        print(f'Build MMCV ({os.cpu_count()} CPU). Có thể mất lâu...')
        build_env = dict(CUDA_HOME=cuda_home, MMCV_WITH_OPS='1', MAX_JOBS=str(os.cpu_count() or 2),
                         PATH=f"{cuda_home}/bin:{os.environ.get('PATH', '')}", TORCH_CUDA_ARCH_LIST='7.5;8.0')
        wheel_tmp = T/'wheelhouse'
        ch.run([PY, '-m', 'pip', 'wheel', '-v', '--no-build-isolation', '--no-deps', '-w', wheel_tmp, src], LOG,
               env=build_env, title='build MMCV wheel')
        built = sorted(wheel_tmp.glob('mmcv-*.whl'))
        if not built:
            fail('Build MMCV không tạo ra wheel; xem ' + str(LOG))
        (OUT/'wheels').mkdir(exist_ok=True)
        shutil.copy2(built[-1], OUT/'wheels'/built[-1].name)
        print('Đã lưu wheel để tải về:', OUT/'wheels'/built[-1].name)
        pip('--no-deps', str(built[-1]))
    if not mmcv_ok():
        fail('MMCV chưa import được mmcv.ops.nms; xem ' + str(LOG))

if not (MOCR/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEYOCRV2_URL'], MOCR], LOG)
ch.run(['git', '-C', MOCR, 'checkout', '--detach', PIN['MONKEYOCRV2_COMMIT']], LOG)
addon = MOCR/'detection/mmocr'
mmocr_root = addon/'mmocr'
backbone_file = mmocr_root/'mmocr/models/common/backbones/monkeyocr_v2_vitae.py'
mmocr_loc = ch.output_of([PY, '-c', 'import mmocr,os;print(os.path.dirname(mmocr.__file__))'])
if not (ver('mmocr') == PIN['MMOCR'] and backbone_file.is_file() and mmocr_loc and str(mmocr_root) in mmocr_loc):
    if not (mmocr_root/'.git').is_dir():
        ch.run(['git', 'clone', '--branch', 'v1.0.1', '--depth', '1', 'https://github.com/open-mmlab/mmocr.git', mmocr_root], LOG)
    setup_path = mmocr_root/'setup.py'
    setup = setup_path.read_text()
    old_exec = "        exec(compile(f.read(), version_file, 'exec'))"
    new_exec = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)"
    if old_exec in setup:
        setup = setup.replace(old_exec, new_exec)
        setup = setup.replace("locals()['short_version']", "namespace['short_version']").replace("locals()['__version__']", "namespace['__version__']")
        setup_path.write_text(setup)
    elif new_exec not in setup:
        fail('MMOCR setup.py khác cấu trúc đã kiểm.')
    ch.run(['bash', 'install.sh'], LOG, cwd=addon, title='MonkeyOCRv2 detection/mmocr/install.sh',
           env=dict(PATH=f"{Path(PY).parent}:{os.environ.get('PATH', '')}", PIP_VERBOSE='1'))
if ver('Pillow') != PIN['PILLOW']:
    pip(f"Pillow=={PIN['PILLOW']}")
if ver('opencv-python') != PIN['OPENCV'] or any(ver(n) for n in ('opencv-python-headless', 'opencv-contrib-python')):
    ch.run([PY, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'], LOG)
    pip('--force-reinstall', '--no-deps', f"opencv-python=={PIN['OPENCV']}")
if ver('shapely') is None:
    pip('shapely')

VERIFY = """
import hashlib, importlib.metadata as m, importlib.util, json, torch
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
from mmcv.ops import nms
register_all_modules(init_default_scope=True)
spec = importlib.util.find_spec('mmocr.models.textdet.postprocessors.db_postprocessor')
out = dict(versions={k: m.version(k) for k in ('torch','torchvision','mmengine','mmcv','mmdet','mmocr','Pillow','opencv-python','numpy','shapely')},
           cuda=torch.cuda.is_available(), device_count_visible=torch.cuda.device_count(), device=torch.cuda.get_device_name(0),
           backbone_registered=MODELS.get('MonkeyOCRv2ViTAEBackbone') is not None,
           db_postprocessor_sha256=hashlib.sha256(open(spec.origin,'rb').read()).hexdigest())
print(json.dumps(out))
"""
raw = ch.output_of([PY, '-c', VERIFY], env=ENV)
if not raw:
    fail('Kiểm cuối môi trường lỗi khi import; xem ' + str(LOG))
env_report = json.loads(raw.splitlines()[-1])
(OUT/f'env_{GPU}.json').write_text(json.dumps(dict(env_report, pins=PIN, gpus=gpus, config=CONFIG_NOTE), indent=1))
print(json.dumps(env_report, indent=1))
problems = []
if env_report['db_postprocessor_sha256'] != PIN['DB_PP_SHA256']:
    problems.append('DBPostprocessor khác bản Tuấn ghim')
if not env_report['backbone_registered']:
    problems.append('MonkeyOCRv2ViTAEBackbone chưa đăng ký')
if env_report['versions']['Pillow'] != PIN['PILLOW']:
    problems.append('Pillow không phải 11.3.0 (cần để tạo LR2 đúng byte)')
if env_report['device_count_visible'] != 1:
    problems.append('Tiến trình đo thấy nhiều hơn 1 GPU')
if problems:
    fail('Môi trường chưa đạt: ' + '; '.join(problems))
print('\nÔ 2 xong.')

## Ô 3 · Repo của Tuấn, config, 2 checkpoint, 2.262 ảnh Clean

Clone `MONKEY_V2` @ `dc19b85`, vá config (chỉ đường dẫn encoder), kiểm SHA 2 checkpoint. Dùng 2.262 ảnh Clean có sẵn trong dataset (kiểm SHA từng ảnh). Chỉ khi dataset thiếu ảnh mới tải zip TextVQA chính thức (7,07 GB, chưa đo thời gian). Sau đó chạy test CPU.

In [ ]:
# ===================== Ô 3 · DỮ LIỆU =====================
from pathlib import PureWindowsPath
# Dataset remains read-only and hash-verified. Patch only the working code copy
# so existing Windows-generated manifests work on Linux; keep manifest bytes intact.
patch_receipt = {}
for rel in ('mixab/dualagree_janek.py', 'mixab/merge_audit_janek.py'):
    target = CODE/rel
    original = (DS/'code'/rel).read_text(encoding='utf-8')
    updated = original.replace('from pathlib import Path\n', 'from pathlib import Path, PureWindowsPath\n')
    for expr in ("model['checkpoint']", "by_alpha[alpha]['checkpoint']"):
        updated = updated.replace(f'Path({expr}).name', f'PureWindowsPath({expr}).name')
    compile(updated, str(target), 'exec')
    target.write_text(updated, encoding='utf-8')
    patch_receipt[rel] = dict(dataset_sha256=sha256(DS/'code'/rel), executed_sha256=sha256(target))
(OUT/'checkpoint_path_compatibility.json').write_text(json.dumps(patch_receipt, indent=2), encoding='utf-8')
LOG = LOGS/f'data_{GPU}.log' 
if not (MV2/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEY_V2_URL'], MV2], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
ch.run(['git', '-C', MV2, 'checkout', '--detach', PIN['MONKEY_V2_COMMIT']], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
if ch.output_of(['git', '-C', MV2, 'status', '--porcelain', '--untracked-files=no']):
    fail('Checkout MONKEY_V2 bị sửa.')
if sha256(BUNDLE/'bundle_manifest.json') != PIN['BUNDLE_MANIFEST_SHA256']:
    fail('bundle_manifest.json khác bản đã khóa.')
CONFIG_PATH.write_text(ch.patch_inference_config((MV2/'docs/handoff/inference_config.py').read_text(encoding='utf-8'), ENCODER), encoding='utf-8')
m = json.loads((WEIGHTS/'merge_manifest.json').read_text())
missing = [PureWindowsPath(x['checkpoint']).name for x in m['models']
           if x['alpha'] in ('0', '1') and not (WEIGHTS/PureWindowsPath(x['checkpoint']).name).is_file()]
if missing:
    fail(f'Dataset thiếu checkpoint: {missing}')
for x in m['models']:
    if x['alpha'] in ('0', '1') and sha256(WEIGHTS/PureWindowsPath(x['checkpoint']).name) != x['checkpoint_sha256']:
        fail(f"{PureWindowsPath(x['checkpoint']).name} sai SHA.")
# Ảnh Clean: tải zip TextVQA chính thức (7,07 GB) vào /tmp, chỉ trích 2.262 ảnh, kiểm SHA từng ảnh với source_cohort.csv
items = ch.needed_clean_images(BUNDLE)
CLEAN = DS/'clean'                                   # ưu tiên ảnh đã có trong dataset (đúng byte từ cache Drive)
check = ch.verify_images(CLEAN, items)
if check['ok'] != len(items):
    print('Dataset không có đủ ảnh Clean đúng SHA:', check, '-> tải zip TextVQA chính thức.')
    CLEAN = T/'textvqa'
    check = ch.verify_images(CLEAN, items)
if check['ok'] != len(items):
    big = T/'downloads/train_val_images.zip'
    big.parent.mkdir(parents=True, exist_ok=True)
    URL, BYTES = 'https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip', 7072297970
    if not (big.is_file() and big.stat().st_size == BYTES):
        ch.run(['wget', '-c', '-q', '-O', big, URL], LOG, title='download TextVQA zip (7 GB)')
    if big.stat().st_size != BYTES:
        fail(f'Zip TextVQA có {big.stat().st_size} byte, kỳ vọng {BYTES}; chạy lại ô để tải tiếp.')
    ch.extract_members(big, [r for r, _ in items], CLEAN)
    check = ch.verify_images(CLEAN, items)
    if check['ok'] == len(items):
        big.unlink()
print('Ảnh Clean:', check)
if check['ok'] != 2262:
    fail('Ảnh Clean chưa đủ hoặc sai SHA.')
ch.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], LOG, cwd=CODE, env=ENV, title='mixab unit tests')
print('\nÔ 3 xong.')

## Ô 4 · Probe 40 nguồn Clean: thời gian + độ lệch so với A2r (A100)

So counts tính lại từ dump với counts Clean của A2r ở cả 72 setting. T4 khác A100 nên dự kiến có lệch ±1 count; ô chỉ báo tỉ lệ, không chặn. Ô cũng in thời gian dự kiến cho dump đầy đủ.

In [ ]:
# ===================== Ô 4 · PROBE =====================
PROBE = OUT/'clean_probe'
if not (PROBE/'dump_done.json').is_file():
    ch.run([PY, '-m', 'mixab.dualagree_janek', 'dump', '--condition', 'clean', '--bundle', BUNDLE, '--clean-root', CLEAN,
            '--prepared-root', T/'prepared', '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--probe', '40',
            '--chunk', '16', '--output', PROBE], LOGS/'clean_probe.log', cwd=CODE, env=ENV, title='clean probe dump')
done = json.loads((PROBE/'dump_done.json').read_text())
for name, p in done['profile'].items():
    print(f"  {name:8} {p['mean_seconds_per_task']:.2f} s/ảnh → dump đủ 2.262 ảnh ~{p['mean_seconds_per_task']*2262/60:.0f} phút")
proc = subprocess.run([PY, '-m', 'mixab.dualagree_janek', 'fidelity', '--dump', str(PROBE), '--bundle', str(BUNDLE), '--npz-run', str(A2R)],
                      cwd=CODE, env=dict(os.environ, **ENV), capture_output=True, text=True)
if proc.returncode != 0 or '{' not in proc.stdout:
    print(proc.stdout[-3000:]); print(proc.stderr[-4000:]); fail('Kiểm trung thực lỗi.')
fid = json.loads(proc.stdout[proc.stdout.index('{'):])
(PROBE/'fidelity.json').write_text(json.dumps(fid, indent=1))
for name, f in fid.items():
    print(f"  {name:8} so với A2r Clean (A100): {f['status']} ({f['mismatched']}/{f['checked']} ô lệch)")

## Ô 5 · Dump đủ 2.262 ảnh Clean × 2 model (GPU)

Lưu theo shard 64 ảnh trong `cs431_out/clean_dump/`. Bị ngắt trong cùng phiên thì chạy lại ô này, shard xong sẽ bỏ qua.

In [ ]:
# ===================== Ô 5 · DUMP CLEAN =====================
CDUMP = OUT/'clean_dump'
if not (CDUMP/'dump_done.json').is_file():
    ch.run([PY, '-m', 'mixab.dualagree_janek', 'dump', '--condition', 'clean', '--bundle', BUNDLE, '--clean-root', CLEAN,
            '--prepared-root', T/'prepared', '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--chunk', '64',
            '--output', CDUMP], LOGS/f'clean_dump_{time.strftime("%Y%m%d-%H%M%S")}.log', cwd=CODE, env=ENV, title='clean full dump')
print(json.loads((CDUMP/'dump_done.json').read_text())['profile'])

## Ô 6 · Đánh giá CS-B1 trên Clean + hệ thống không định tuyến (CPU)

In [ ]:
# ===================== Ô 6 · ĐÁNH GIÁ =====================
EV = OUT/'csb1_clean_eval'
ch.run([PY, '-m', 'mixab.csb1_clean_janek', '--bundle', BUNDLE, '--clean-dump', OUT/'clean_dump', '--lr2-dump', DUMP, '--output', EV],
       LOGS/'csb1_clean_eval.log', cwd=CODE, env=ENV, title='csb1 clean evaluate')
r = json.loads((EV/'csb1_clean_result.json').read_text())
for sp in ('ALL', 'A', 'B'):
    e = r['splits'][sp]
    print(f"== {sp} ({e['sources']} nguồn)")
    for arm in e['mean']:
        print(f"  {arm:16} Clean {e['clean'][arm]['hmean']*100:.2f}  LR2 {e['lr2'][arm]['hmean']*100:.2f}  mean {e['mean'][arm]*100:.2f}")
    d = e['CSB1_minus_B1 .40/.60']
    print(f"  CS-B1 − B1 .40/.60: Clean {d['clean']['delta_pp']:+.2f} {d['clean']['ci95_pp']}, LR2 {d['lr2']['delta_pp']:+.2f} {d['lr2']['ci95_pp']}")
    print(f"  hệ thống không định tuyến − model đơn tốt nhất ({e['best_single_by_mean']}): {e['router_free_system_minus_best_single_pp']:+.2f} pp")
print('Cổng nhóm trên A (mô tả):', r['group_gate_A_descriptive'])

## Ô 7 · Đóng gói

Tạo `/kaggle/working/cs431_csb1_clean_<GPU>_<thời gian>.zip`. Tải ở khung **Output** và gửi lại Claude (hoặc báo Claude tải bằng Kaggle API).

In [ ]:
# ===================== Ô 7 · ĐÓNG GÓI =====================
pack = W/f'cs431_csb1_clean_{GPU}_{time.strftime("%Y%m%d-%H%M%S")}.zip'
with zipfile.ZipFile(pack, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix != '.whl':
            zf.write(p, p.relative_to(OUT).as_posix())
    zf.writestr('code_manifest_sha256.txt', code_sha + '\n')
print('Đã đóng gói:', pack, f'({pack.stat().st_size/2**20:.1f} MiB)')